# Models

CMR and memory search model variants


This section documents the memory search models available in jaxcmr. The library provides a unified framework for implementing and comparing different variants of the Context Maintenance and Retrieval (CMR) model.

## Why Do We Model Memory?

Free recall experiments reveal robust patterns:

- **Recency**: Items from the end of a list are recalled first and most often
- **Primacy**: Items from the beginning are also recalled well
- **Contiguity**: Recalled items tend to come from nearby serial positions
- **Forward asymmetry**: Transitions favor items that followed the just-recalled item

CMR explains these patterns through a simple mechanism: temporal context that drifts during study and is reinstated during retrieval.

## Model Architecture

A CMR model combines three core components:

- **[Context](context.ipynb)**: A temporal context representation that drifts over time
- **[Memory](linear_memory.ipynb)**: Associative stores (MFC and MCF) linking items and contexts
- **Termination Policy**: Determines when recall stops

These components implement protocols defined in `jaxcmr.typing`, allowing different implementations to be swapped while maintaining compatibility with the rest of the library.

## Baseline Models

### [CMR](cmr.ipynb)
The base Context Maintenance and Retrieval model. Start here to understand the core mechanism that all variants build upon.

### [Context](context.ipynb)
The TemporalContext component implements drifting context representations that form the heart of CMR's temporal dynamics.

### [Memory](linear_memory.ipynb)
Linear (Hebbian) and Instance (MINERVA-2 style) memory implementations, with tradeoffs between event-blending and event-preserving storage.

### [Parameters](parameters.ipynb)
Complete reference for all CMR parameters, their meanings, and typical value ranges.

## Repetition Variants

Models addressing how repeated items are encoded and retrieved:

| Model | Description |
|-------|-------------|
| [Positional CMR](repetition/positional.ipynb) | Position-based encoding creates distinct traces for repetitions |
| [Drift Positional](repetition/drift.ipynb) | Different drift rate for repeated items |
| [Reinforcement](repetition/reinforcement.ipynb) | Reinforces first-presentation trace at repetition |
| [Outlist CMR](repetition/outlist.ipynb) | Out-of-list context mixing for distinct repetition traces |
| [Blend Positional](repetition/blend.ipynb) | Hybrid item/position context streams |
| [Distinct Contexts](repetition/distinct.ipynb) | Positional encoding with item-level recallability |
| [No Reinstate](repetition/no-reinstate.ipynb) | No context reinstatement during encoding |

## Semantic Variants

Models integrating pre-experimental semantic associations:

| Model | Description |
|-------|-------------|
| [Additive Semantic](semantic/additive.ipynb) | Semantic similarity as additive boost to activation |
| [Multiplicative Semantic](semantic/multiplicative.ipynb) | Semantic as multiplicative gate (foraging-inspired) |

<!-- TODO: CRU variants (cru/) and EEG variants (eeg/) — models exist in code but templates not yet created -->

## Usage Example

In [ ]:
from jaxcmr.models.cmr import CMR, make_factory
import jaxcmr.components.linear_memory as LinearMemory
import jaxcmr.components.context as TemporalContext
from jaxcmr.components.termination import PositionalTermination

# Create a model factory
Factory = make_factory(
    LinearMemory.init_mfc,
    LinearMemory.init_mcf,
    TemporalContext.init,
    PositionalTermination,
)

factory = Factory(dataset, features=None)

# Define parameters
params = {
    "encoding_drift_rate": 0.5,
    "start_drift_rate": 0.5,
    "recall_drift_rate": 0.5,
    "learning_rate": 0.5,
    "primacy_scale": 2.0,
    "primacy_decay": 0.8,
    "shared_support": 0.05,
    "item_support": 0.25,
    "choice_sensitivity": 1.0,
    "stop_probability_scale": 0.05,
    "stop_probability_growth": 0.2,
    "learn_after_context_update": True,
    "allow_repeated_recalls": False,
}

# Create model for a specific trial
model = factory.create_trial_model(trial_index=0, parameters=params)

# Study items (1-indexed)
for item in [1, 2, 3, 4, 5]:
    model = model.experience(item)

# Begin retrieval
model = model.start_retrieving()

# Get retrieval probabilities
probs = model.outcome_probabilities()

## Choosing a Model

### For Standard Free Recall
Start with the base [CMR](cmr.ipynb) model. It captures recency, primacy, contiguity, and forward asymmetry with a minimal parameter set.

### For Lists with Repeated Items
Consider [Positional CMR](repetition/positional.ipynb) or [Blend Positional CMR](repetition/blend.ipynb). These handle the ambiguity of which presentation's context to reinstate.

### For Semantic Recall
Use [Additive Semantic CMR](semantic/additive.ipynb) for typical semantic effects, or [Multiplicative Semantic CMR](semantic/multiplicative.ipynb) for foraging-inspired gating behavior.

## Implementation Philosophy

All models in jaxcmr share a common interface defined in `jaxcmr.typing.MemorySearch`:

- `experience(item)` — Process a study event
- `start_retrieving()` — Transition to retrieval mode
- `retrieve(item)` — Process a recall event
- `outcome_probabilities()` — Get retrieval probability distribution

This allows models to be swapped in evaluation pipelines without changing the analysis code.